# Week 6, Lab 4 — Two MCP servers, one agent


In [1]:
import zipfile
import os

zip_path = "/content/shared.zip"      # Path of the uploaded ZIP file
extract_path = "/content/shared"      # Folder where files will be extracted

# Create the folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ ZIP extracted successfully!")
print("Files extracted to:", extract_path)

✅ ZIP extracted successfully!
Files extracted to: /content/shared


In [2]:
import zipfile
import os

zip_path = "/content/shared.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully!")

✅ Extracted successfully!


In [3]:
WEEK = 'Week 6'
LAB = 'Lab 4 — multiple servers'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 6 / Lab 4 — multiple servers
Environment: Google Colab
Backend: huggingface
Tip: Runtime → Change runtime type → T4 GPU for faster generation.
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [4]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn mcp
else:
    %pip install -q mcp ollama


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.9 MB/s eta 0:00:00


In [6]:
# ==========================
# Fake MCP Servers for Colab
# ==========================

# ----- Tools Server -----
def calculator(expression: str):
    return str(eval(expression))

def lookup_fact(topic: str):
    facts = {
        "mcp": "Model Context Protocol (MCP) is a standard that connects AI models with tools and external data sources.",
        "langgraph": "LangGraph builds stateful multi-agent workflows using graphs.",
        "ollama": "Ollama lets you run open-source LLMs locally on your machine."
    }
    return facts.get(topic.lower(), "No fact found.")

TOOLS_SERVER = {
    "calculator": calculator,
    "lookup_fact": lookup_fact,
}

# ----- Notes Server -----
notes_db = []

def add_note(text: str):
    notes_db.append(text)
    return f"Saved note: {text}"

def list_notes():
    return notes_db

NOTES_SERVER = {
    "add_note": add_note,
    "list_notes": list_notes,
}

# ==========================
# MCP-style call() function
# ==========================

async def call(server: str, tool: str, args: dict):
    if server == "tools":
        return TOOLS_SERVER[tool](**args)

    elif server == "notes":
        return NOTES_SERVER[tool](**args)

    else:
        return "Unknown server."

# ==========================
# Test (same as V-Align)
# ==========================

print("calc ->", await call("tools", "calculator", {"expression": "2+2"}))

print("note ->", await call(
    "notes",
    "add_note",
    {"text": "MCP servers are just processes."}
))

print("list ->", await call("notes", "list_notes", {}))

calc -> 4
note -> Saved note: MCP servers are just processes.
list -> ['MCP servers are just processes.']


Write a ReAct loop that picks (server, tool) from a merged catalog.
